In [63]:
import pandas as pd
import numpy as np

In [64]:
name = input("Введите название инструмента(пример: AFKS, GOLD, YDEX):")

In [65]:
df: pd.DataFrame = pd.read_csv(f"/Users/side/Desktop/Trading Chaos AI/df/data/{name}_20.csv")


In [66]:
df.info ()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66875 entries, 0 to 66874
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   DateTime         66875 non-null  object 
 1   Open             66875 non-null  float64
 2   High             66875 non-null  float64
 3   Low              66875 non-null  float64
 4   Close            66875 non-null  float64
 5   Volume           66875 non-null  int64  
 6   Alligator_Jaw    66875 non-null  float64
 7   Alligator_Teeth  66875 non-null  float64
 8   Alligator_Lips   66875 non-null  float64
 9   Fractal_Up       8956 non-null   float64
 10  Fractal_Down     8969 non-null   float64
 11  AO               66875 non-null  float64
dtypes: float64(10), int64(1), object(1)
memory usage: 6.1+ MB


Смотрим данные на наличие пропусков

In [67]:
for col in df.columns:
    pct_missing = df[col].isnull().mean()
    print('{} - {}%'.format(col, round(pct_missing*100)))

DateTime - 0%
Open - 0%
High - 0%
Low - 0%
Close - 0%
Volume - 0%
Alligator_Jaw - 0%
Alligator_Teeth - 0%
Alligator_Lips - 0%
Fractal_Up - 87%
Fractal_Down - 87%
AO - 0%


в фракталах добавим бинарность, тк много пропусков

1 вход для фрактала лонг
-1 вход для фрактала шорт

0 отсутствие входа

In [68]:
df['Fractal_Down'] = df['Fractal_Down'].notna().astype(int)
df['Fractal_Up'] = df['Fractal_Up'].notna().astype(int)

for col in df.columns:
    pct_missing = df[col].isnull().mean()
    print('{} - {}%'.format(col, round(pct_missing*100)))

DateTime - 0%
Open - 0%
High - 0%
Low - 0%
Close - 0%
Volume - 0%
Alligator_Jaw - 0%
Alligator_Teeth - 0%
Alligator_Lips - 0%
Fractal_Up - 0%
Fractal_Down - 0%
AO - 0%


In [69]:
df. describe( include='all'). T

/Users/side/Desktop/Trading Chaos AI/.venv/lib/python3.14/site-packages/numpy/core/_methods.py:49: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
/Users/side/Desktop/Trading Chaos AI/.venv/lib/python3.14/site-packages/numpy/core/_methods.py:49: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
/Users/side/Desktop/Trading Chaos AI/.venv/lib/python3.14/site-packages/numpy/core/_methods.py:49: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
DateTime,66875,66875,2025.12.20 18:00,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Open,66875.0,NaN,NaN,NaN,1564.5495,627.051384,568.0,1214.4,1365.4,1798.0,4386.7
High,66875.0,NaN,NaN,NaN,1567.071225,627.971329,574.0,1216.1,1367.9,1800.65,4398.4
Low,66875.0,NaN,NaN,NaN,1561.957892,626.077788,567.0,1212.4,1363.1,1795.3,4374.0
Close,66875.0,NaN,NaN,NaN,1564.572321,627.099472,568.1,1214.2,1365.4,1797.95,4386.4
Volume,66875.0,NaN,NaN,NaN,293.535178,1278.963398,1.0,35.0,50.0,60.0,41418.0
Alligator_Jaw,66875.0,NaN,NaN,NaN,inf,inf,580.35251,1214.19933,1363.73691,1798.1933,1797693134862315708145274237317043567980705675...
Alligator_Teeth,66875.0,NaN,NaN,NaN,inf,inf,577.34121,1214.096495,1364.16894,1798.0991,1797693134862315708145274237317043567980705675...
Alligator_Lips,66875.0,NaN,NaN,NaN,1564.119427,626.404893,575.88171,1214.000765,1364.55799,1798.130405,4371.48927
Fractal_Up,66875.0,NaN,NaN,NaN,0.133921,0.34057,0.0,0.0,0.0,0.0,1.0


В классической стратегии нам нет смысла использовать объемы, поэтому убеерем их



In [70]:
df.drop(['Volume'], axis=1, inplace=True)
df

,DateTime,Open,High,Low,Close,Alligator_Jaw,Alligator_Teeth,Alligator_Lips,Fractal_Up,Fractal_Down,AO
0,2025.12.20 18:00,4374.2,4374.3,4372.4,4373.0,4.360489e+03,4.367992e+03,4371.48927,0,0,8.533530e+00
1,2025.12.20 17:00,4373.6,4374.7,4373.0,4374.1,4.359567e+03,4.367263e+03,4371.16159,0,0,9.160000e+00
2,2025.12.20 16:00,4372.9,4373.7,4372.3,4373.6,4.358781e+03,4.366550e+03,4370.75199,0,0,9.582650e+00
3,2025.12.20 15:00,4373.0,4373.0,4372.6,4372.7,4.357550e+03,4.365671e+03,4370.16499,0,0,1.018000e+01
4,2025.12.20 14:00,4373.3,4373.3,4372.3,4372.6,4.356126e+03,4.364832e+03,4369.64373,0,0,1.046382e+01
...,...,...,...,...,...,...,...,...,...,...,...
66870,2006.06.14 18:00,574.0,574.0,574.0,574.0,1.797693e+308,5.973813e+02,596.95325,0,1,1.797693e+308
66871,2006.06.14 17:00,575.7,575.7,575.7,575.7,1.797693e+308,1.797693e+308,599.44157,0,0,1.797693e+308
66872,2006.06.14 16:00,585.7,585.7,575.7,575.7,1.797693e+308,1.797693e+308,597.07696,0,0,1.797693e+308
66873,2006.06.14 15:00,587.0,587.0,587.0,587.0,1.797693e+308,1.797693e+308,599.87120,0,0,1.797693e+308


Cделаем новый столбец "Color AO" - тоже сделаем бинарный:

- 1 зеленый - лонговый  

- -1 красный - шортовый

In [71]:
df['Color AO'] = df['AO'].diff().apply(lambda x: 1 if x > 0 else (-1 if x < 0 else 0))
df['Color AO'] = df['Color AO'].fillna(0).astype(int)

In [72]:
print(df[['AO', 'Color AO']].head(15))
print("\nРаспределение цветов баров:")
print(df['Color AO'].value_counts().sort_index())

          AO  Color AO
0    8.53353         0
1    9.16000         1
2    9.58265         1
3   10.18000         1
4   10.46382         1
5   10.27588        -1
6   11.33059         1
7   12.91206         1
8   15.02147         1
9   16.41176         1
10  17.50059         1
11  15.71382        -1
12  13.82294        -1
13  10.14206        -1
14   8.11588        -1

Распределение цветов баров:
Color AO
-1    33151
 0       31
 1    33693
Name: count, dtype: int64


добавляем перемменную и признаки для Аллигатора

первый код = "что есть сигнал?"


In [73]:
# сортируем по дате на всякий случай
df = df.sort_values("DateTime").reset_index(drop=True)


Состояние Аллигатора

In [74]:
jaw, teeth, lips = df["Alligator_Jaw"], df["Alligator_Teeth"], df["Alligator_Lips"]

bullish = (lips > teeth) & (teeth > jaw) # бычий тренд
bearish = (jaw  > teeth) & (teeth > lips) # медвежий тренд

df["Alligator_Bullish"] = bullish.astype(int)
df["Alligator_Bearish"] = bearish.astype(int)

df["AlligatorStart_Long"]  = (bullish & ~bullish.shift(1, fill_value=False)).astype(int)
df["AlligatorStart_Short"] = (bearish & ~bearish.shift(1, fill_value=False)).astype(int)

Логика по АО

In [75]:
df["AO_sign"] = np.where(df["Color AO"] > 0, 1, -1)

df["AO_zero_up"]   = ((df["AO"] > 0) & (df["AO"].shift(1) <= 0)).astype(int)
df["AO_zero_down"] = ((df["AO"] < 0) & (df["AO"].shift(1) >= 0)).astype(int)

# три подряд бара одного цвета
df["AO_three_green"] = (
    (df["AO_sign"]==1) &
    (df["AO_sign"].shift(1)==1) &
    (df["AO_sign"].shift(2)==1)
).astype(int)

df["AO_three_red"] = (
    (df["AO_sign"]==-1) &
    (df["AO_sign"].shift(1)==-1) &
    (df["AO_sign"].shift(2)==-1)
).astype(int)

# блюдце по AO
df["AO_saucer_up"] = (
    (df["AO"] > 0) &
    (df["AO"].shift(2) > df["AO"].shift(1)) &
    (df["AO"] > df["AO"].shift(1))
).astype(int)

df["AO_saucer_down"] = (
    (df["AO"] < 0) &
    (df["AO"].shift(2) < df["AO"].shift(1)) &
    (df["AO"] < df["AO"].shift(1))
).astype(int)

Генерация сигналов входа

In [76]:
df["EntrySignal"] = 0
df["EntryReason"] = 0

state = "flat"
last_alligator_side = 0

цикл по всем барам:

In [77]:
for i in range(len(df)):
    start_long  = bool(df.at[i, "AlligatorStart_Long"])
    start_short = bool(df.at[i, "AlligatorStart_Short"])

    # отслеживаем когда аллигатор открылся вверх/вниз
    if state in ("flat", "in_long", "in_short"):
        if start_long:
            state = "wait_long"; last_alligator_side = 1
        elif start_short:
            state = "wait_short"; last_alligator_side = -1

    # если появился новый «старт» аллигатора, переходим в режим ожидания сигнала AO (wait_long/wait_short)
    if state == "wait_long":
        if df.at[i, "AO_zero_up"]==1:
            df.at[i, "EntrySignal"] = 1; df.at[i, "EntryReason"] = "1"; state = "in_long" # zero line
        elif df.at[i, "AO_three_green"]==1:
            df.at[i, "EntrySignal"] = 1; df.at[i, "EntryReason"] = "2"; state = "in_long" # three colors
        elif df.at[i, "AO_saucer_up"]==1:
            df.at[i, "EntrySignal"] = 1; df.at[i, "EntryReason"] = "3"; state = "in_long" # saucer

    # В бычьем режиме ждём один из трёх паттернов AO 
    # Как только он случился – ставим EntrySignal = 1, записываем причину и считаем, что мы «в лонге»
    elif state == "wait_short":
        if df.at[i, "AO_zero_down"]==1:
            df.at[i, "EntrySignal"] = -1; df.at[i, "EntryReason"] = "1"; state = "in_short"
        elif df.at[i, "AO_three_red"]==1:
            df.at[i, "EntrySignal"] = -1; df.at[i, "EntryReason"] = "2"; state = "in_short"
        elif df.at[i, "AO_saucer_down"]==1:
            df.at[i, "EntrySignal"] = -1; df.at[i, "EntryReason"] = "3"; state = "in_short"
    # То же для шорта
    if state in ("in_long", "wait_long") and start_short:
        state = "wait_short"; last_alligator_side = -1
        # Если во время ожидания/позиции появился противоположный аллигатор 
        # переключаемся на ожидание сигнала в новую сторону.
    if state in ("in_short", "wait_short") and start_long:
        state = "wait_long"; last_alligator_side = 1

/var/folders/34/s31951c50p9f5_w1_s5xj4tc0000gn/T/ipykernel_4864/3454918577.py:27: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.at[i, "EntrySignal"] = -1; df.at[i, "EntryReason"] = "2"; state = "in_short"


Подтверждение фракталов

Фрактал у Вильямса подтверждается спустя 2 бара. Поэтому здесь берётся значение фрактала 2 бара назад и переносится вперёд – чтобы на текущем баре понимать, что фрактал уже подтверждён.

In [78]:
df["Fractal_Up_conf"]   = df["Fractal_Up"].shift(2).fillna(0).astype(int)
df["Fractal_Down_conf"] = df["Fractal_Down"].shift(2).fillna(0).astype(int)

Логика добавления к позиции (AddOn)

In [79]:
df["AddOn_Anchor_Level"] = np.nan
df["AddOn_Anchor_IsUp"]  = np.nan
df["AddOn_Size_Pct"]     = np.nan
waiting_anchor = False
pos_side = 0

# Колонки для «якоря» добавочных входов: 
# уровень цены, направление (1 – вверх, 0 – вниз), размер добавки (в процентах).
for i in range(len(df)):
    sig = int(df.at[i, "EntrySignal"])
    if sig != 0:
        pos_side = sig
        waiting_anchor = True
        continue
    # когда появился вход (EntrySignal ≠ 0) — запоминаем направление позиции и включаем режим waiting_anchor:
    #  ждём подходящего фрактала для постановки уровня добавки
    if waiting_anchor and pos_side == 1 and df.at[i, "Fractal_Up_conf"] == 1:
        df.at[i, "AddOn_Anchor_Level"] = df.at[i, "High"]
        df.at[i, "AddOn_Anchor_IsUp"]  = 1
        df.at[i, "AddOn_Size_Pct"]     = 0.30
        waiting_anchor = False
    # Если в лонге и появился подтверждённый верхний фрактал – его максимум становится AddOn_Anchor_Level,
    #  направление вверх, размер 30% от базовой позиции.
    if waiting_anchor and pos_side == -1 and df.at[i, "Fractal_Down_conf"] == 1:
        df.at[i, "AddOn_Anchor_Level"] = df.at[i, "Low"]
        df.at[i, "AddOn_Anchor_IsUp"]  = 0
        df.at[i, "AddOn_Size_Pct"]     = 0.30
        waiting_anchor = False

Аналогично для шорта (минимум фрактала).

In [80]:
df["AddOn_Anchor_Level"] = df["AddOn_Anchor_Level"].ffill()
df["AddOn_Anchor_IsUp"]  = df["AddOn_Anchor_IsUp"].ffill()
df["AddOn_Size_Pct"]     = df["AddOn_Size_Pct"].ffill()


In [81]:
df

,DateTime,Open,High,Low,Close,Alligator_Jaw,Alligator_Teeth,Alligator_Lips,Fractal_Up,Fractal_Down,...,AO_three_red,AO_saucer_up,AO_saucer_down,EntrySignal,EntryReason,Fractal_Up_conf,Fractal_Down_conf,AddOn_Anchor_Level,AddOn_Anchor_IsUp,AddOn_Size_Pct
0,2006.06.14 13:00,608.9,608.9,608.9,608.9,1.797693e+308,1.797693e+308,596.46400,0,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
1,2006.06.14 15:00,587.0,587.0,587.0,587.0,1.797693e+308,1.797693e+308,599.87120,0,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
2,2006.06.14 16:00,585.7,585.7,575.7,575.7,1.797693e+308,1.797693e+308,597.07696,0,0,...,1,0,0,0,0,0,0,NaN,NaN,NaN
3,2006.06.14 17:00,575.7,575.7,575.7,575.7,1.797693e+308,1.797693e+308,599.44157,0,0,...,1,0,0,0,0,0,0,NaN,NaN,NaN
4,2006.06.14 18:00,574.0,574.0,574.0,574.0,1.797693e+308,5.973813e+02,596.95325,0,1,...,1,0,0,-1,2,0,0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66870,2025.12.20 14:00,4373.3,4373.3,4372.3,4372.6,4.356126e+03,4.364832e+03,4369.64373,0,0,...,0,1,0,0,0,0,0,4339.2,1.0,0.3
66871,2025.12.20 15:00,4373.0,4373.0,4372.6,4372.7,4.357550e+03,4.365671e+03,4370.16499,0,0,...,0,0,0,0,0,0,0,4339.2,1.0,0.3
66872,2025.12.20 16:00,4372.9,4373.7,4372.3,4373.6,4.358781e+03,4.366550e+03,4370.75199,0,0,...,0,0,0,0,0,0,0,4339.2,1.0,0.3
66873,2025.12.20 17:00,4373.6,4374.7,4373.0,4374.1,4.359567e+03,4.367263e+03,4371.16159,0,0,...,0,0,0,0,0,0,0,4339.2,1.0,0.3


второй код = «из чего модель будет его угадывать?».


In [82]:
df = df.copy().sort_values("DateTime").reset_index(drop=True)

# 1) Подтверждённые фракталы (каузально)
df["Fractal_Up_conf"]   = df["Fractal_Up"].shift(2).fillna(0).astype(int)
df["Fractal_Down_conf"] = df["Fractal_Down"].shift(2).fillna(0).astype(int)

# 2) Якорь добора — первый фрактал в сторону позиции ПОСЛЕ входа
df["AddOn_Anchor_Level"] = np.nan
df["AddOn_Anchor_IsUp"]  = np.nan  # 1=верхний, 0=нижний
df["AddOn_Size_Pct"]     = np.nan
df["AddOn_Ready"]        = 0       # «якорь найден, добор ещё не выполнен»
df["AddOn_Triggered"]    = 0       # «добор исполнен по цене якоря»

in_pos = 0           # +1 long, -1 short, 0 flat
anchor_set = False   # найден ли якорь добора для текущей сделки
addon_done = False   # добор уже выполнен?

for i in range(len(df)):
    sig = int(df.at[i, "EntrySignal"]) if "EntrySignal" in df.columns else 0

    # вход открывает новую «сделку»
    if in_pos == 0 and sig != 0:
        in_pos = sig
        anchor_set = False
        addon_done = False
        continue  # на самом баре входа якорь ещё не может появиться (фракталы подтверждаются позже)

    # если в позиции и якорь ещё не выбран — ловим ПЕРВЫЙ подтверждённый фрактал в сторону сделки
    if in_pos == 1 and not anchor_set and df.at[i, "Fractal_Up_conf"] == 1:
        df.at[i, "AddOn_Anchor_Level"] = df.at[i, "High"]
        df.at[i, "AddOn_Anchor_IsUp"]  = 1
        df.at[i, "AddOn_Size_Pct"]     = 0.30
        df.at[i, "AddOn_Ready"]        = 1
        anchor_set = True
    elif in_pos == -1 and not anchor_set and df.at[i, "Fractal_Down_conf"] == 1:
        df.at[i, "AddOn_Anchor_Level"] = df.at[i, "Low"]
        df.at[i, "AddOn_Anchor_IsUp"]  = 0
        df.at[i, "AddOn_Size_Pct"]     = 0.30
        df.at[i, "AddOn_Ready"]        = 1
        anchor_set = True

    # если якорь выбран, проверим срабатывание (по High/Low)
    if anchor_set and not addon_done:
        anchor = df.at[i, "AddOn_Anchor_Level"]
        if in_pos == 1 and not pd.isna(anchor) and df.at[i, "High"] >= anchor:
            df.at[i, "AddOn_Triggered"] = 1
            addon_done = True
        elif in_pos == -1 and not pd.isna(anchor) and df.at[i, "Low"] <= anchor:
            df.at[i, "AddOn_Triggered"] = 1
            addon_done = True

    # выход по развороту аллигатора обнуляет состояние (если у тебя уже есть ExitSignal — можно использовать его)
    # Здесь считаем flip как смену устойчивого порядка:
    jaw, teeth, lips = df["Alligator_Jaw"], df["Alligator_Teeth"], df["Alligator_Lips"]
    bullish = (lips > teeth) & (teeth > jaw)
    bearish = (jaw  > teeth) & (teeth > lips)
    start_long  = bool(bullish.iloc[i] and not bullish.shift(1, fill_value=False).iloc[i])
    start_short = bool(bearish.iloc[i] and not bearish.shift(1, fill_value=False).iloc[i])

    if in_pos == 1 and start_short:
        in_pos = 0; anchor_set = False; addon_done = False
    elif in_pos == -1 and start_long:
        in_pos = 0; anchor_set = False; addon_done = False

# протащим постоянные значения якоря вперёд до конца сделки (удобно для исполнителя)
df["AddOn_Anchor_Level"] = df["AddOn_Anchor_Level"].ffill()
df["AddOn_Anchor_IsUp"]  = df["AddOn_Anchor_IsUp"].ffill()
df["AddOn_Size_Pct"]     = df["AddOn_Size_Pct"].ffill()


In [83]:
df

,DateTime,Open,High,Low,Close,Alligator_Jaw,Alligator_Teeth,Alligator_Lips,Fractal_Up,Fractal_Down,...,AO_saucer_down,EntrySignal,EntryReason,Fractal_Up_conf,Fractal_Down_conf,AddOn_Anchor_Level,AddOn_Anchor_IsUp,AddOn_Size_Pct,AddOn_Ready,AddOn_Triggered
0,2006.06.14 13:00,608.9,608.9,608.9,608.9,1.797693e+308,1.797693e+308,596.46400,0,0,...,0,0,0,0,0,NaN,NaN,NaN,0,0
1,2006.06.14 15:00,587.0,587.0,587.0,587.0,1.797693e+308,1.797693e+308,599.87120,0,0,...,0,0,0,0,0,NaN,NaN,NaN,0,0
2,2006.06.14 16:00,585.7,585.7,575.7,575.7,1.797693e+308,1.797693e+308,597.07696,0,0,...,0,0,0,0,0,NaN,NaN,NaN,0,0
3,2006.06.14 17:00,575.7,575.7,575.7,575.7,1.797693e+308,1.797693e+308,599.44157,0,0,...,0,0,0,0,0,NaN,NaN,NaN,0,0
4,2006.06.14 18:00,574.0,574.0,574.0,574.0,1.797693e+308,5.973813e+02,596.95325,0,1,...,0,-1,2,0,0,NaN,NaN,NaN,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66870,2025.12.20 14:00,4373.3,4373.3,4372.3,4372.6,4.356126e+03,4.364832e+03,4369.64373,0,0,...,0,0,0,0,0,4339.2,1.0,0.3,0,0
66871,2025.12.20 15:00,4373.0,4373.0,4372.6,4372.7,4.357550e+03,4.365671e+03,4370.16499,0,0,...,0,0,0,0,0,4339.2,1.0,0.3,0,0
66872,2025.12.20 16:00,4372.9,4373.7,4372.3,4373.6,4.358781e+03,4.366550e+03,4370.75199,0,0,...,0,0,0,0,0,4339.2,1.0,0.3,0,0
66873,2025.12.20 17:00,4373.6,4374.7,4373.0,4374.1,4.359567e+03,4.367263e+03,4371.16159,0,0,...,0,0,0,0,0,4339.2,1.0,0.3,0,0


Предобработка временных меток в DataFrame

In [84]:
df = df.copy().sort_values("DateTime").reset_index(drop=True)
df["DateTime"] = pd.to_datetime(df["DateTime"], errors="coerce")
df = df[df["DateTime"].notna()].reset_index(drop=True)


In [85]:
df

,DateTime,Open,High,Low,Close,Alligator_Jaw,Alligator_Teeth,Alligator_Lips,Fractal_Up,Fractal_Down,...,AO_saucer_down,EntrySignal,EntryReason,Fractal_Up_conf,Fractal_Down_conf,AddOn_Anchor_Level,AddOn_Anchor_IsUp,AddOn_Size_Pct,AddOn_Ready,AddOn_Triggered
0,2006-06-14 13:00:00,608.9,608.9,608.9,608.9,1.797693e+308,1.797693e+308,596.46400,0,0,...,0,0,0,0,0,NaN,NaN,NaN,0,0
1,2006-06-14 15:00:00,587.0,587.0,587.0,587.0,1.797693e+308,1.797693e+308,599.87120,0,0,...,0,0,0,0,0,NaN,NaN,NaN,0,0
2,2006-06-14 16:00:00,585.7,585.7,575.7,575.7,1.797693e+308,1.797693e+308,597.07696,0,0,...,0,0,0,0,0,NaN,NaN,NaN,0,0
3,2006-06-14 17:00:00,575.7,575.7,575.7,575.7,1.797693e+308,1.797693e+308,599.44157,0,0,...,0,0,0,0,0,NaN,NaN,NaN,0,0
4,2006-06-14 18:00:00,574.0,574.0,574.0,574.0,1.797693e+308,5.973813e+02,596.95325,0,1,...,0,-1,2,0,0,NaN,NaN,NaN,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66870,2025-12-20 14:00:00,4373.3,4373.3,4372.3,4372.6,4.356126e+03,4.364832e+03,4369.64373,0,0,...,0,0,0,0,0,4339.2,1.0,0.3,0,0
66871,2025-12-20 15:00:00,4373.0,4373.0,4372.6,4372.7,4.357550e+03,4.365671e+03,4370.16499,0,0,...,0,0,0,0,0,4339.2,1.0,0.3,0,0
66872,2025-12-20 16:00:00,4372.9,4373.7,4372.3,4373.6,4.358781e+03,4.366550e+03,4370.75199,0,0,...,0,0,0,0,0,4339.2,1.0,0.3,0,0
66873,2025-12-20 17:00:00,4373.6,4374.7,4373.0,4374.1,4.359567e+03,4.367263e+03,4371.16159,0,0,...,0,0,0,0,0,4339.2,1.0,0.3,0,0


In [86]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66875 entries, 0 to 66874
Data columns (total 32 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   DateTime              66875 non-null  datetime64[ns]
 1   Open                  66875 non-null  float64       
 2   High                  66875 non-null  float64       
 3   Low                   66875 non-null  float64       
 4   Close                 66875 non-null  float64       
 5   Alligator_Jaw         66875 non-null  float64       
 6   Alligator_Teeth       66875 non-null  float64       
 7   Alligator_Lips        66875 non-null  float64       
 8   Fractal_Up            66875 non-null  int64         
 9   Fractal_Down          66875 non-null  int64         
 10  AO                    66875 non-null  float64       
 11  Color AO              66875 non-null  int64         
 12  Alligator_Bullish     66875 non-null  int64         
 13  Alligator_Bearis

In [87]:
df['EntryReason'] = df['EntryReason'].astype('int32')

In [88]:
for col in df.columns:
    pct_missing = df[col].isnull().mean()
    print('{} - {}%'.format(col, round(pct_missing*100)))

DateTime - 0%
Open - 0%
High - 0%
Low - 0%
Close - 0%
Alligator_Jaw - 0%
Alligator_Teeth - 0%
Alligator_Lips - 0%
Fractal_Up - 0%
Fractal_Down - 0%
AO - 0%
Color AO - 0%
Alligator_Bullish - 0%
Alligator_Bearish - 0%
AlligatorStart_Long - 0%
AlligatorStart_Short - 0%
AO_sign - 0%
AO_zero_up - 0%
AO_zero_down - 0%
AO_three_green - 0%
AO_three_red - 0%
AO_saucer_up - 0%
AO_saucer_down - 0%
EntrySignal - 0%
EntryReason - 0%
Fractal_Up_conf - 0%
Fractal_Down_conf - 0%
AddOn_Anchor_Level - 0%
AddOn_Anchor_IsUp - 0%
AddOn_Size_Pct - 0%
AddOn_Ready - 0%
AddOn_Triggered - 0%


In [89]:
df.describe(include='all').T

/Users/side/Desktop/Trading Chaos AI/.venv/lib/python3.14/site-packages/numpy/core/_methods.py:49: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
/Users/side/Desktop/Trading Chaos AI/.venv/lib/python3.14/site-packages/numpy/core/_methods.py:49: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
/Users/side/Desktop/Trading Chaos AI/.venv/lib/python3.14/site-packages/numpy/core/_methods.py:49: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)


,count,mean,min,25%,50%,75%,max,std
DateTime,66875,2016-10-05 06:23:06.491214848,2006-06-14 13:00:00,2012-02-02 18:30:00,2016-11-07 12:00:00,2021-07-07 12:30:00,2025-12-20 18:00:00,NaN
Open,66875.0,1564.5495,568.0,1214.4,1365.4,1798.0,4386.7,627.051384
High,66875.0,1567.071225,574.0,1216.1,1367.9,1800.65,4398.4,627.971329
Low,66875.0,1561.957892,567.0,1212.4,1363.1,1795.3,4374.0,626.077788
Close,66875.0,1564.572321,568.1,1214.2,1365.4,1797.95,4386.4,627.099472
Alligator_Jaw,66875.0,inf,580.35251,1214.19933,1363.73691,1798.1933,1797693134862315708145274237317043567980705675...,inf
Alligator_Teeth,66875.0,inf,577.34121,1214.096495,1364.16894,1798.0991,1797693134862315708145274237317043567980705675...,inf
Alligator_Lips,66875.0,1564.119427,575.88171,1214.000765,1364.55799,1798.130405,4371.48927,626.404893
Fractal_Up,66875.0,0.133921,0.0,0.0,0.0,0.0,1.0,0.34057
Fractal_Down,66875.0,0.134116,0.0,0.0,0.0,0.0,1.0,0.340779


In [90]:
df['EntrySignal'].value_counts()

EntrySignal
 0    63506
 1     1694
-1     1675
Name: count, dtype: int64

In [91]:
df['EntryReason'].value_counts()

EntryReason
0    63506
2     2435
3      826
1      108
Name: count, dtype: int64

In [92]:
def build_target(df: pd.DataFrame, h: int = 20, target_type: str = "classification") -> pd.DataFrame:

    df = df.copy()

    df["Close_fwd"] = df["Close"].shift(-h)

    df["ret_H"] = np.where(
        df["EntrySignal"] > 0,
        (df["Close_fwd"] - df["Close"]) / df["Close"],

        np.where(
            df["EntrySignal"] < 0,
            (df["Close"] - df["Close_fwd"]) / df["Close"],
            0.0
        )
    )

    if target_type == "classification":
        df["GoodTrade"] = ((df["EntrySignal"] != 0) & (df["ret_H"] > 0)).astype(int)

    df = df.dropna(subset=["Close_fwd"])

    return df
df = build_target(df, h=20, target_type="classification")

In [93]:
df.to_csv(f"/Users/side/Desktop/Trading Chaos AI/df/clean_df/{name}.csv", index=False)